# Example 41 — CIFAR-100 Compression Benchmark

Compares five storage/loader combinations for CIFAR-100 **train** split:

| Case | Storage | Loader |
|------|---------|--------|
| **torch-raw** | torchvision pickle files | `torch.utils.data.DataLoader` |
| **vvtk-none+torch** | `VVTKDataset` `compression=['none','none']` | `torch.utils.data.DataLoader` |
| **vvtk-zstd+torch** | `VVTKDataset` `compression=['zstd','zstd']` | `torch.utils.data.DataLoader` |
| **vvtk-none+vvtkdl** | `VVTKDataset` `compression=['none','none']` | `VVTKDataLoader` (C++) |
| **vvtk-zstd+vvtkdl** | `VVTKDataset` `compression=['zstd','zstd']` | `VVTKDataLoader` (C++) |

Metrics reported per case:
- **Disk size** (MB, relative to torch-raw)
- **1-epoch loading time** (seconds, relative to torch-raw)

All data is stored locally in `./data/`.

In [1]:
import os
import time
import shutil
import glob
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from vvtk_dataset import VVTKDataset, VVTKDataLoader

DATA_DIR       = './data'
RAW_DIR        = os.path.join(DATA_DIR, 'cifar100_raw')   # torchvision download root
VVTK_NONE_DIR  = os.path.join(DATA_DIR, 'train_none')     # VVTK uncompressed
VVTK_ZSTD_DIR  = os.path.join(DATA_DIR, 'train_zstd')     # VVTK zstd

NUM_SHARDS  = 16
BATCH_SIZE  = 256
NUM_WORKERS = 4  

os.makedirs(DATA_DIR, exist_ok=True)
print(f'Data root  : {os.path.abspath(DATA_DIR)}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Num workers: {NUM_WORKERS}')

Data root  : /extra3/cadrete/work/src/vvtk_dataset/examples/example41_compression_benchmark/data
Batch size : 256
Num workers: 4


## Step 1 — Download CIFAR-100

In [2]:
train_cifar = datasets.CIFAR100(root=RAW_DIR, train=True, download=True)

print(f'Train samples: {len(train_cifar)}')
print(f'Image shape (HWC): {np.array(train_cifar[0][0]).shape}')

Train samples: 50000
Image shape (HWC): (32, 32, 3)


## Step 2 — Write VVTK datasets

In [3]:
def write_vvtk(torchvision_ds, path, compression, force=False):
    """Write a CIFAR torchvision dataset to a VVTK path.
    Tensors: img uint8 [3,32,32], label int64 [1]
    """
    master = path + '.master'
    if not force and os.path.exists(master):
        print(f'  [skip] {path} already exists')
        return
    # Remove any partial shards
    for f in glob.glob(path + '.*'):
        os.remove(f)
    t0 = time.time()
    with VVTKDataset(path, mode='wb', num_shards=NUM_SHARDS, compression=compression) as ds:
        for idx in range(len(torchvision_ds)):
            img_pil, label = torchvision_ds[idx]
            img = np.array(img_pil, dtype=np.uint8).transpose(2, 0, 1)  # CHW
            img = np.ascontiguousarray(img)
            lbl = np.array([label], dtype=np.int64)
            ds.add(idx, img, lbl)
    elapsed = time.time() - t0
    print(f'  Written {len(torchvision_ds)} samples → {path}  ({elapsed:.1f}s)')


print('Writing vvtk-none ...')
write_vvtk(train_cifar, VVTK_NONE_DIR, compression=['none', 'none'])

print('Writing vvtk-zstd ...')
write_vvtk(train_cifar, VVTK_ZSTD_DIR, compression=['zstd', 'zstd'])

Writing vvtk-none ...
  [skip] ./data/train_none already exists
Writing vvtk-zstd ...
  [skip] ./data/train_zstd already exists


## Step 3 — Measure disk sizes

In [4]:
def dir_size_mb(path):
    """Total size of a directory tree in MB."""
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total / (1024 ** 2)


def vvtk_size_mb(path_prefix):
    """Total size of all shard files + master file."""
    total = sum(os.path.getsize(f) for f in glob.glob(path_prefix + '.*'))
    return total / (1024 ** 2)


size_raw  = dir_size_mb(os.path.join(RAW_DIR, 'cifar-100-python'))  # extracted tarball
size_none = vvtk_size_mb(VVTK_NONE_DIR)
size_zstd = vvtk_size_mb(VVTK_ZSTD_DIR)

print(f'Disk usage (train split only):')
print(f'  torch-raw  (cifar-100-python/train) : {size_raw:8.2f} MB')
print(f'  vvtk-none                           : {size_none:8.2f} MB')
print(f'  vvtk-zstd                           : {size_zstd:8.2f} MB')
print()
print(f'  zstd vs raw     : {100*(1 - size_zstd/size_raw):+.1f}% size change')
print(f'  zstd vs no-comp : {100*(1 - size_zstd/size_none):+.1f}% size change')

Disk usage (train split only):
  torch-raw  (cifar-100-python/train) :   177.67 MB
  vvtk-none                           :   153.83 MB
  vvtk-zstd                           :   141.89 MB

  zstd vs raw     : +20.1% size change
  zstd vs no-comp : +7.8% size change


## Step 4 — Define DataLoaders

In [ ]:
# --- 1. Reference: standard torchvision dataset ---
class TorchCIFAR100(torch.utils.data.Dataset):
    def __init__(self, root):
        self.ds = datasets.CIFAR100(root=root, train=True, download=False,
                                    transform=transforms.ToTensor())
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        return self.ds[idx]


# --- 2. VVTK wrapper (for standard torch DataLoader) ---
class VVTKCIFARDataset(torch.utils.data.Dataset):
    def __init__(self, path, compression):
        self.ds = VVTKDataset(path, mode='rb', compression=compression)
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        img, lbl = self.ds[idx]
        return img.float().div(255.0), lbl.squeeze().long()


torch_ds      = TorchCIFAR100(RAW_DIR)
vvtk_none_ds  = VVTKCIFARDataset(VVTK_NONE_DIR, compression=['none', 'none'])
vvtk_zstd_ds  = VVTKCIFARDataset(VVTK_ZSTD_DIR, compression=['zstd', 'zstd'])

def make_torch_loader(ds):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=False,
                      persistent_workers=(NUM_WORKERS > 0))

torch_loader     = make_torch_loader(torch_ds)
vvtk_none_loader = make_torch_loader(vvtk_none_ds)
vvtk_zstd_loader = make_torch_loader(vvtk_zstd_ds)

# --- 3. VVTKDataLoader (C++ loader with zstd decompression support) ---
# img: uint8 [3,32,32],  lbl: int64 [1]
_cifar_shapes = [(3, 32, 32), (1,)]
_cifar_dtypes = [torch.uint8, torch.int64]

vvtk_none_dl = VVTKDataLoader(
    vvtk_none_ds.ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shapes=_cifar_shapes,
    dtypes=_cifar_dtypes,
    shuffle=False,
)

vvtk_zstd_dl = VVTKDataLoader(
    vvtk_zstd_ds.ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shapes=_cifar_shapes,
    dtypes=_cifar_dtypes,
    shuffle=False,
)

print(f'torch-raw       batches/epoch: {len(torch_loader)}')
print(f'vvtk-none+torch batches/epoch: {len(vvtk_none_loader)}')
print(f'vvtk-zstd+torch batches/epoch: {len(vvtk_zstd_loader)}')
print(f'vvtk-none+vvtkdl batches/epoch: {len(vvtk_none_dl)}')
print(f'vvtk-zstd+vvtkdl batches/epoch: {len(vvtk_zstd_dl)}')

[VVTKLoader] Building index map...
[VVTKLoader] Ready in 0.01s
[VVTKLoader] Building index map...
[VVTKLoader] Ready in 0.01s
torch-raw       batches/epoch: 196
vvtk-none+torch batches/epoch: 196
vvtk-zstd+torch batches/epoch: 196
vvtk-none+vvtkdl batches/epoch: 196
vvtk-zstd+vvtkdl batches/epoch: 196


: 

## Step 5 — Measure 1-epoch DataLoader throughput

In [ ]:
def time_one_epoch(loader, label):
    """Iterate a standard torch DataLoader once."""
    total_samples = 0
    t0 = time.time()
    for imgs, lbls in loader:
        total_samples += imgs.size(0)
    elapsed = time.time() - t0
    throughput = total_samples / elapsed
    print(f'  {label:<26s}: {elapsed:6.2f}s  ({throughput:,.0f} samples/s)')
    return elapsed


def time_one_epoch_vvtk(loader, label):
    """Iterate a VVTKDataLoader once.
    Each batch is ((img_tensor, img_len), (lbl_tensor, lbl_len)).
    """
    total_samples = 0
    t0 = time.time()
    for batch in loader:
        imgs = batch[0][0]   # [B, 3, 32, 32] uint8
        total_samples += imgs.size(0)
    elapsed = time.time() - t0
    throughput = total_samples / elapsed
    print(f'  {label:<26s}: {elapsed:6.2f}s  ({throughput:,.0f} samples/s)')
    return elapsed


print('1-epoch DataLoader timing (shuffle=False, pin_memory=False):')
t_torch          = time_one_epoch(torch_loader,      'torch-raw')
t_none           = time_one_epoch(vvtk_none_loader,  'vvtk-none+torch')
t_zstd           = time_one_epoch(vvtk_zstd_loader,  'vvtk-zstd+torch')
t_none_vvtk_dl   = time_one_epoch_vvtk(vvtk_none_dl, 'vvtk-none+vvtkdl')
t_zstd_vvtk_dl   = time_one_epoch_vvtk(vvtk_zstd_dl, 'vvtk-zstd+vvtkdl')

1-epoch DataLoader timing (shuffle=False, pin_memory=False):


## Step 6 — Summary table

In [ ]:
results = {
    'torch-raw'        : {'size_mb': size_raw,  'epoch_s': t_torch},
    'vvtk-none+torch'  : {'size_mb': size_none, 'epoch_s': t_none},
    'vvtk-zstd+torch'  : {'size_mb': size_zstd, 'epoch_s': t_zstd},
    'vvtk-none+vvtkdl' : {'size_mb': size_none, 'epoch_s': t_none_vvtk_dl},
    'vvtk-zstd+vvtkdl' : {'size_mb': size_zstd, 'epoch_s': t_zstd_vvtk_dl},
}

ref_size = results['torch-raw']['size_mb']
ref_time = results['torch-raw']['epoch_s']

hdr = f'{"Case":<22} {"Size (MB)":>10} {"vs raw":>8} {"Epoch (s)":>10} {"vs raw":>8}'
print(hdr)
print('-' * len(hdr))
for case, v in results.items():
    size_pct = 100 * (v['size_mb'] / ref_size - 1)
    time_pct = 100 * (v['epoch_s'] / ref_time - 1)
    print(f"{case:<22} {v['size_mb']:>10.1f} {size_pct:>+7.1f}% {v['epoch_s']:>10.2f} {time_pct:>+7.1f}%")

Case            Size (MB)   vs raw  Epoch (s)   vs raw
--------------------------------------------------------
torch-raw           177.7    +0.0%       1.24    +0.0%
vvtk-none           153.8   -13.4%       1.13    -8.3%
vvtk-zstd           141.9   -20.1%       1.55   +25.2%
